In [2]:
import numpy as np
from zfish._io import import16chFlt
import matplotlib.pyplot as plt
from tqdm import tqdm
from zfish.utils import *
from zfish.swim import *
from zfish.local_path import *

code_id = "5005 - Linear Track Passive Navigation"
loc = join(figpath, code_id)
mkdir(loc)
pass

        D:\Yao Shuyang\En Lab\Results\5005 - Linear Track Passive Navigation is already existed!


In [ ]:
dirs = [
    #r"D:\EnData\Light-sheet\10128\res.16chFlt",
    # r"D:\EnData\Light-sheet\10129\res.16chFlt",
    #r"D:\EnData\Light-sheet\10130\res.16chFlt",
    #r"D:\EnData\Light-sheet\10130\S2\res.16chFlt",
    #r"D:\EnData\Light-sheet\10131\S1\res.16chFlt",
    #r"D:\EnData\Light-sheet\10131\S2\res.16chFlt",
    #r"D:\EnData\Light-sheet\10132\res.16chFlt",
    #r"D:\EnData\Light-sheet\10132\S2\res.16chFlt",
    r"D:\EnData\10135\S1\res.16chFlt",
]
titles = [
    #"10128",
    #"10129",
    #"10130",
    #"10130-S2",
    #"10131-S1",
    #"10131-S2",
    #"10132-S1",
    #"10132-S2",
    "10135-S1"
]
for dir, title, i in zip(dirs, titles, range(len(dirs))):
    if i <= -1:
        continue
        
        res = import16chFlt(dir, 16) 
        res['behav_pos_y'][(res['behav_pos_y'] < 6)] = np.nan
        idx = np.where(np.isnan(res['behav_pos_y']) == False)[0]
        res['behav_pos_y'][(res['behav_pos_y'] >105)] = np.nan
        fig, axes = plt.subplots(ncols=2, nrows=1, figsize=(10, 3), gridspec_kw={'width_ratios': [4, 1]})
        ax = Clear_Axes(axes[0], close_spines=['top', 'right'], ifxticks=True, ifyticks=True)
        ax.plot(res["behav_time"][idx], res["behav_pos_y"][idx])
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Position (%)")
        ax.set_title(title)
        ax1 = Clear_Axes(axes[1], close_spines=['top', 'right'], ifxticks=True, ifyticks=True)
        itip = np.where(np.diff(res["behav_pos_y"][idx]) > 10)[0]
        beg = np.concatenate(([0], itip + 1))
        end = np.concatenate((itip+1, [len(idx)]))
        for j in range(len(beg)):
            segment_idx = idx[beg[j]:end[j]][np.where(res['behav_speed_y'][idx[beg[j]:end[j]]] > 0)[0]]
            dt = np.diff(res['behav_time'][segment_idx])
            dx = np.diff(res['behav_pos_y'][segment_idx])
            v = dx / dt
            v[(dx > 10) | (dx < 0)] = np.nan 
            v[np.isnan(v) == False] = np.convolve(v[np.isnan(v) == False], np.ones(1000)/1000, mode='same')
            ax1.plot(v, res['behav_pos_y'][segment_idx[:-1]], color='gray', linewidth=0.12)
        ax1.set_xlabel("Speed (%)")
        ax1.set_ylabel("Position (%)")
        ax1.set_ylim(ax.get_ylim())
        ax1.set_xlim(0, 10)
        ax1.set_xticks(np.linspace(0, 10, 6))
        plt.show()
        print(np.nanmax(res["behav_time"][idx]))
    else:
        res = import16chFlt(dir, 21)
        print(np.unique(res['map']))
        res['behav_pos_y'][(res['behav_pos_y'] >105)] = np.nan
        idxnan = np.where(np.isnan(res['behav_pos_y']) == True)[0]
        for k in res.keys():
            res[k] = np.delete(res[k], idxnan)

        idx = np.arange(len(res['behav_pos_y']))    
        fig, axes = plt.subplots(ncols=2, nrows=1, figsize=(10, 3), gridspec_kw={'width_ratios': [4, 1]})
        ax = Clear_Axes(axes[0], close_spines=['top', 'right'], ifxticks=True, ifyticks=True)
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Position (%)")
        ax.set_title(title)
        ax1 = Clear_Axes(axes[1], close_spines=['top', 'right'], ifxticks=True, ifyticks=True)
        itip = np.where(np.diff(res["behav_pos_y"]) < -5)[0]
        beg = np.concatenate(([0], itip + 1))
        end = np.concatenate((itip+1, [len(res['behav_pos_y'])]))
        for j in range(len(beg)):
            segment_idx = np.where(res['behav_speed_y'][beg[j]:end[j]] > 0)[0]
            dt = np.diff(res['behav_time'][beg[j]:end[j]][segment_idx])
            dx = np.diff(res['behav_pos_y'][beg[j]:end[j]][segment_idx])
            filtered_idx = np.where((dx > 10) | (dx < 0))[0]
            v = np.convolve(dx, np.ones(600), mode='same') / np.convolve(dt, np.ones(600), mode='same')
            v[(dx > 10) | (dx < 0)] = np.nan 
            colors = ['','#333766', '', '#A4C096']
            nmap = res['map'][beg[j]:end[j]][segment_idx[-10]]-1
            ax.plot(res['behav_time'][beg[j]:end[j]][segment_idx[:-1]], res['behav_pos_y'][beg[j]:end[j]][segment_idx[:-1]], color=colors[nmap], linewidth=0.5)
            ax1.plot(v, res['behav_pos_y'][beg[j]:end[j]][segment_idx[:-1]], color='gray', linewidth=0.12)
        ax1.set_xlabel("Speed (%)")
        ax1.set_ylabel("Position (%)")
        ax1.set_ylim(ax.get_ylim())
        ax1.set_xlim(0, 10)
        ax1.set_xticks(np.linspace(0, 10, 6))
        plt.show()

In [ ]:
with open(r"D:\EnData\10135\S1\trace.pkl", 'rb') as f:
    trace = pickle.load(f)
    
idx = np.where((trace['SI'][:, 0] >= 1)&(np.sum(trace['Spikes'], axis=1) > 10))[0]

def rate_map(trace, cell_idx):
    save_dir = join(trace['save_dir'], "RateMap")
    mkdir(save_dir)
    idx = np.where(trace['spike_nodes'] >= 0)[0]
    
    fig = plt.figure(figsize=(3, 2))
    ax = Clear_Axes(plt.axes(), close_spines=['top', 'right'], ifxticks=True, ifyticks=True)
    for i in tqdm(cell_idx):
        sns.lineplot(
            x=trace['spike_nodes'][idx],
            y=trace['RawTraces'][i, idx],
            hue=trace['ms_map'][idx],
            palette=['#333766', '#A4C096'],
            linewidth=0.5,
            ax=ax,
            legend=False,
            err_kws={'edgecolor': None}
        )
        ax.set_xlim(-0.5, 50.5)
        ax.set_xlabel("Position (%)")
        ax.set_ylabel("dF/F")
        ax.set_title(f"Cell {i}")
        ax.set_xticks(np.linspace(0, 50, 6), np.linspace(0, 100, 6).astype(int))
        ymin, ymax = ax.get_ylim()
        yymax = max(np.abs(ymin)*3, np.abs(ymax))
        ax.set_ylim(-yymax/3, yymax)
        plt.tight_layout()
        plt.savefig(join(save_dir, f"Cell_{i}.png"), dpi=300)
        
        ax.clear()
    plt.close(fig)

def time_coding_map(trace, cell_idx):
    save_dir = join(trace['save_dir'], "TimeCell")
    mkdir(save_dir)
    
    fig = plt.figure(figsize=(3, 2))
    ax = Clear_Axes(plt.axes(), close_spines=['top', 'right'], ifxticks=True, ifyticks=True)
    t = (trace['ms_time_aligned']//500).astype(np.float64)/2
    for i in tqdm(cell_idx):
        sns.lineplot(
            x=t,
            y=trace['RawTraces'][i, :],
            hue=trace['ms_map'],
            palette=['#333766', '#A4C096'],
            linewidth=0.5,
            ax=ax,
            legend=False,
            err_kws={'edgecolor': None}
        )
        ax.set_xlim(-0.5, 50.5)
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("dF/F")
        ax.set_title(f"Cell {i}")
        ymin, ymax = ax.get_ylim()
        ax.set_xlim(-10, 30)
        ax.set_xticks(np.linspace(-10, 30, 9).astype(int))
        yymax = max(np.abs(ymin)*3, np.abs(ymax))
        ax.set_ylim(-yymax/3, yymax)
        plt.tight_layout()
        plt.savefig(join(save_dir, f"Cell_{i}.png"), dpi=300)
        
        ax.clear()
    plt.close(fig)


def LinearizedMap(
    trace,
    cell_idx
):
    
    save_dir = join(trace['save_dir'], "LinearizedMap")
    mkdir(save_dir)
    
    fig = plt.figure(figsize=(8, 2))
    ax = Clear_Axes(plt.axes(), close_spines=['top', 'right'], ifxticks=True, ifyticks=True)
    beg, end = trace['lap_beg_idx'], trace['lap_end_idx']
    map_id = trace['map']
    behav_pos = trace['behav_pos']
    behav_time = trace['behav_time']
    for j in range(len(beg)):
        ax.plot(
            behav_time[beg[j]:end[j]]/1000,
            behav_pos[beg[j]:end[j]],
            color='#333766' if map_id[j] == 2 else '#A4C096',
            linewidth=0.5
        )
    lap_beg_time = trace['lap beg time']
    lap_end_time = trace['lap end time']
    indices = []
    for j in range(len(lap_beg_time)):
        idx = np.where(
            (trace['ms_time'] >= lap_beg_time[j]) &
            (trace['ms_time'] <= lap_end_time[j])
        )[0]
        indices.append(idx)
    
    axt = Clear_Axes(ax.twinx(), close_spines=['top', 'left'], ifxticks=True, ifyticks=True)
    within_trial_idx = np.where(trace['spike_nodes'] >= 0)[0]
    for i in tqdm(cell_idx):
        spike_times = trace['ms_time']/1000
        dFF = trace['RawTraces'][i]
        
        a = []
        for j in range(len(lap_beg_time)):
            idx = indices[j]
            b = axt.plot(
                spike_times[idx],
                dFF[idx],  
                color='#333766' if map_id[j] == 2 else '#A4C096', 
                linewidth=0.2
            )
            a += b
    
        ax.set_ylim(-1, 100)
        ax.set_yticks(np.linspace(0, 100, 6))
        ax.set_ylabel("Position (%)")
        ax.set_xlabel("Time (s)")
        axt.set_ylabel("dF/F")
        max = np.nanmax(dFF[within_trial_idx])
        min = np.nanmin(dFF[within_trial_idx])
        axt.set_ylim(min*1.05, max*1.05)
        plt.savefig(join(save_dir, f"Cell_{i}.png"), dpi=300)
        for m in a:
            m.remove()
    plt.close(fig)

time_coding_map(trace,  np.arange(trace['n_neuron']))

        D:\EnData\10135\S1\TimeCell is already existed!


  0%|          | 3/10502 [00:12<12:03:55,  4.14s/it]

In [4]:
# Cells:
a = [933, 921, 906, 874, 799, 813, 819, 863, 893, 914, 875, 846, 744, 746, 708, 701, 700, 707, 743, 723, 822, 1042, 1062, 1075, 1071, 1084, 1098, 1125, 1122, 
 1120, 1098, 1084, 1071, 1085, 1121, 1145, 1156, 1154, 1152, 1452, 1471, 1460, 1478, 1482, 1487, 1496, 1513, 1519, 1528, 1533, 1530, 1557, 1565, 1550,
 1525, 1509, 1498, 1485, 1452, 1477, 1534, 1559, 1568, 1573, 1598, 1587, 1600, 1599, 1654, 1674, 1690, 1706, 1708, 1723, 1736, 1760, 1751, 1789, 1780, 
 1781, 1771, 1735, 1617, 1615, 1630, 1661, 1905, 1912, 1927, 1913, 1911, 1899, 1910, 1920, 1932, 1904, 1918, 1926, 1925, 1922, 1921, 1959, 2003, 2021,
 2046, 2057, 2027, 2042, 2016, 2026, 2006, 1988, 1976, 1971, 1957, 1935, 1940, 2140, 2116, 2170, 2220, 2210, 2228, 2254, 2259, 2267, 2264, 2249, 2119,
 2101, 2090, 2085, 2112, 2199, 2485, 2444, 2437, 2455, 2462, 2488, 2502, 2512, 2539, 2469, 2420, 2418, 2409, 2411, 2410, 2417, 2406, 2437, 2444, 2485,
 2631, 2664, 2674, 2656, 2649, 2566, 2614, 2623, 2643, 2671, 2666, 2768, 2794, 2812, 2819, 2816, 2809, 2795, 2712, 2702, 2698, 2692, 2690, 2674, 2664, 
 2656, 2649, 2689, 2987, 2986, 2979, 3000, 2970, 2961, 2964, 2954, 2953, 2967, 2951, 2963, 2959, 2985, 2983, 3005, 2996, 3050, 3061, 3059, 3080, 3098,
 3075, 3086, 3026, 3025, 3034, 3048, 3169, 3191, 3199, 3128, 3143, 3152, 3148, 3141, 3121, 3214, 3302, 3298, 3337, 3300, 3264, 3270, 3243, 3237, 3534,
 3522, 3531, 3528, 3655, 3558, 3719, 3829, 4233, 4280, 4277, 4286, 4294, 4322, 4325, 4318, 4313, 4612, 4700, 4706, 4659, 4717, 4820, 4824, 4890, 4881,
 5358]
print(sorted(a))

# 重点
[1122, 1145, 1154, 1152, 1452, 1460, 1478, 1513, 1557, 1565, 1587, 1654, 1674, 1789, 1630, 1661, 1959, 2027, 2042, 1976, 1971, 2116, 2140, 2228, 2254,
 2485, 2488, 2512, 2485, 2953, 2963, 3169, 3199, 3143, 3148, 4233, 4280, 4286, 4890]

[700, 701, 707, 708, 723, 743, 744, 746, 799, 813, 819, 822, 846, 863, 874, 875, 893, 906, 914, 921, 933, 1042, 1062, 1071, 1071, 1075, 1084, 1084, 1085, 1098, 1098, 1120, 1121, 1122, 1125, 1145, 1152, 1154, 1156, 1452, 1452, 1460, 1471, 1477, 1478, 1482, 1485, 1487, 1496, 1498, 1509, 1513, 1519, 1525, 1528, 1530, 1533, 1534, 1550, 1557, 1559, 1565, 1568, 1573, 1587, 1598, 1599, 1600, 1615, 1617, 1630, 1654, 1661, 1674, 1690, 1706, 1708, 1723, 1735, 1736, 1751, 1760, 1771, 1780, 1781, 1789, 1899, 1904, 1905, 1910, 1911, 1912, 1913, 1918, 1920, 1921, 1922, 1925, 1926, 1927, 1932, 1935, 1940, 1957, 1959, 1971, 1976, 1988, 2003, 2006, 2016, 2021, 2026, 2027, 2042, 2046, 2057, 2085, 2090, 2101, 2112, 2116, 2119, 2140, 2170, 2199, 2210, 2220, 2228, 2249, 2254, 2259, 2264, 2267, 2406, 2409, 2410, 2411, 2417, 2418, 2420, 2437, 2437, 2444, 2444, 2455, 2462, 2469, 2485, 2485, 2488, 2502, 2512, 2539, 2566, 2614, 2623, 2631, 2643, 2649, 2649, 2656, 2656, 2664, 2664, 2666, 2671, 2674, 2674, 2689, 

[1122,
 1145,
 1154,
 1152,
 1452,
 1460,
 1478,
 1513,
 1557,
 1565,
 1587,
 1654,
 1674,
 1789,
 1630,
 1661,
 1959,
 2027,
 2042,
 1976,
 1971,
 2116,
 2140,
 2228,
 2254,
 2485,
 2488,
 2512,
 2485,
 2953,
 2963,
 3169,
 3199,
 3143,
 3148,
 4233,
 4280,
 4286,
 4890]